####Data Analyst Interview Preparation – SQL & Python Exercises

#####Description:
- This workbook provides junior data analysts with practical SQL and Python exercises commonly asked in interviews. It includes examples on salary analysis, window functions, palindrome and factorial programs, punch-in/out tracking, and calculating total hours worked. Each task demonstrates real-world scenarios and problem-solving techniques to enhance interview readiness.

##### Task 1: 

Find the second highest salary from the Employee table for each Department ( 2 Query)


In [0]:
%sql
CREATE OR REPLACE TABLE Employee (
  S_NO INT,
  NAME STRING,
  DEPARTMENT_ID INT,
  SALARY INT
);

INSERT INTO Employee VALUES
(1, 'Alice', 101, 120000),
(2, 'Bob', 101, 80000),
(3, 'Charlie', 102, 150000),
(4, 'David', 102, 50000),
(5, 'Eve', 103, 130000),
(6, 'Frank', 103, 70000),
(7, 'Grace', 104, 110000),
(8, 'Henry', 104, 90000),
(9, 'Irene', 105, 190000),
(10, 'Jack', 105, 10000);

SELECT * FROM Employee;

S_NO,NAME,DEPARTMENT_ID,SALARY
1,Alice,101,120000
2,Bob,101,80000
3,Charlie,102,150000
4,David,102,50000
5,Eve,103,130000
6,Frank,103,70000
7,Grace,104,110000
8,Henry,104,90000
9,Irene,105,190000
10,Jack,105,10000


In [0]:
%sql

-- 1st way

WITH SECOND_HIGHEST_SALARY AS (
    SELECT NAME,
          SALARY,
          DEPARTMENT_ID,
          DENSE_RANK() OVER (PARTITION BY DEPARTMENT_ID ORDER BY SALARY DESC) AS RANK
    FROM Employee)
  
SELECT DEPARTMENT_ID,
       NAME,
       SALARY
  FROM second_highest_salary
  WHERE RANK = 2
  ORDER  BY 3 DESC;

DEPARTMENT_ID,NAME,SALARY
104,Henry,90000
101,Bob,80000
103,Frank,70000
102,David,50000
105,Jack,10000


In [0]:
%sql
-- 2nd Way
SELECT 
    E.DEPARTMENT_ID,
    E.NAME,
    E.SALARY
FROM EMPLOYEE E
WHERE E.SALARY = (
    SELECT MAX(F.SALARY)
    FROM EMPLOYEE F
    WHERE F.DEPARTMENT_ID = E.DEPARTMENT_ID
      AND F.SALARY < (
          SELECT MAX(SALARY)
          FROM EMPLOYEE
          WHERE DEPARTMENT_ID = F.DEPARTMENT_ID
      )
)
  ORDER BY 3 DESC;

DEPARTMENT_ID,NAME,SALARY
104,Henry,90000
101,Bob,80000
103,Frank,70000
102,David,50000
105,Jack,10000



#####Task 2:

Write an SQL query to generate a department-wise salary report with the following requirements:

1. Calculate the total salary paid in each department.
2. Identify the employee(s) who have the highest salary in each department.
3. For each employee, compute their salary as a percentage of the total salary of their respective department.
4. Create a column optimization_potential:
Return 'YES' if the employee’s salary contribution is greater than 50% of the department total
Otherwise return 'NO'
5. Create a column highest_salary_flag:
Return 'HIGHEST' for employees with the maximum salary in their department
Otherwise return 'NORMAL'

In [0]:
%sql

WITH BASE_TABLE AS (
    SELECT 
        S_NO, 
        NAME,
        DEPARTMENT_ID,
        SALARY,
        SUM(SALARY) OVER (PARTITION BY DEPARTMENT_ID) AS TOTAL_SALARY_PER_DEPARTMENT
    FROM EMPLOYEE
)

SELECT 
    S_NO,
    NAME, 
    DEPARTMENT_ID,
    SALARY, 
    TOTAL_SALARY_PER_DEPARTMENT,

    CONCAT(
        ROUND(SALARY * 100.0 / TOTAL_SALARY_PER_DEPARTMENT, 1),
        '%'
    ) AS SALARY_PERCENTAGE,

    CASE 
        WHEN SALARY * 100.0 / TOTAL_SALARY_PER_DEPARTMENT < 50 THEN 'NO'
        ELSE 'YES'
    END AS OPTIMIZATION_POTENTIAL,

    CASE 
        WHEN SALARY = MAX(SALARY) OVER (PARTITION BY DEPARTMENT_ID) 
        THEN 'HIGHEST'
        ELSE 'NORMAL'
    END AS HIGHEST_SALARY_PER_DEPARTMENT

FROM BASE_TABLE;

S_NO,NAME,DEPARTMENT_ID,SALARY,TOTAL_SALARY_PER_DEPARTMENT,SALARY_PERCENTAGE,OPTIMIZATION_POTENTIAL,HIGHEST_SALARY_PER_DEPARTMENT
1,Alice,101,120000,200000,60.0%,YES,HIGHEST
2,Bob,101,80000,200000,40.0%,NO,NORMAL
3,Charlie,102,150000,200000,75.0%,YES,HIGHEST
4,David,102,50000,200000,25.0%,NO,NORMAL
5,Eve,103,130000,200000,65.0%,YES,HIGHEST
6,Frank,103,70000,200000,35.0%,NO,NORMAL
7,Grace,104,110000,200000,55.0%,YES,HIGHEST
8,Henry,104,90000,200000,45.0%,NO,NORMAL
9,Irene,105,190000,200000,95.0%,YES,HIGHEST
10,Jack,105,10000,200000,5.0%,NO,NORMAL



#####Python

######Task 3:

Write a Python function that takes a string as input and checks whether it is a palindrome or not.

- A palindrome is a word that reads the same forward and backward (e.g., "madam", "racecar").
Return True if the input is a palindrome, otherwise return False.

In [0]:

def palindrome2(name):
    s = str(name)
    return s == s[::-1]

name = input("Enter a Name: ")
print(palindrome2(name))

Enter a Name:  racecar

True



#####Task 4:

Write a Python function to calculate the factorial of a given number using recursion.

- Factorial of a number n is the product of all positive integers less than or equal to n.

- Example: 5! = 5 × 4 × 3 × 2 × 1 = 120
The function should take an integer input and return its factorial.

In [0]:
def fact(n):
    if n==0 or n==1:
        return 1
    else:
        return n * fact(n-1)

n = int(input("Enter a number: "))
print("Factorial:", fact(n))

Enter a number:  5

Factorial: 120


###### Task 5:

Write an SQL query to identify employees whose most recent punch record indicates they are currently inside the office.


In [0]:
%sql

CREATE OR REPLACE TABLE Punch_Log (
    ID INT,
    EMPLOYEE_ID INT,
    PUNCH_TIME TIMESTAMP,
    STATUS STRING
);

INSERT INTO Punch_Log VALUES
(1, 101, '2025-06-25 08:00:00', 'IN'),
(2, 102, '2025-06-25 08:15:00', 'IN'),
(3, 101, '2025-06-25 17:00:00', 'OUT'),
(4, 103, '2025-06-25 09:00:00', 'IN'),
(5, 102, '2025-06-25 12:00:00', 'OUT'),
(6, 104, '2025-06-25 09:30:00', 'IN'),
(7, 105, '2025-06-25 10:00:00', 'IN'),
(8, 105, '2025-06-25 16:30:00', 'OUT'),
(9, 104, '2025-06-25 18:00:00', 'OUT'),
(10, 103, '2025-06-25 18:15:00', 'OUT'),
(11, 103, '2025-06-26 08:31:00', 'IN');

SELECT * 
FROM Punch_Log
ORDER BY 2;

ID,EMPLOYEE_ID,PUNCH_TIME,STATUS
1,101,2025-06-25T08:00:00.000Z,IN
3,101,2025-06-25T17:00:00.000Z,OUT
2,102,2025-06-25T08:15:00.000Z,IN
5,102,2025-06-25T12:00:00.000Z,OUT
4,103,2025-06-25T09:00:00.000Z,IN
11,103,2025-06-26T08:31:00.000Z,IN
10,103,2025-06-25T18:15:00.000Z,OUT
6,104,2025-06-25T09:30:00.000Z,IN
9,104,2025-06-25T18:00:00.000Z,OUT
7,105,2025-06-25T10:00:00.000Z,IN


In [0]:
%sql

WITH EMP_STATUS AS (
    SELECT ID,
       EMPLOYEE_ID,
       PUNCH_TIME,
       STATUS, 
       ROW_NUMBER() OVER (PARTITION BY EMPLOYEE_ID ORDER BY PUNCH_TIME DESC) AS NEXT_ENTRY
    FROM PUNCH_LOG )

SELECT EMPLOYEE_ID,
       PUNCH_TIME,
       STATUS
FROM EMP_STATUS
WHERE NEXT_ENTRY = 1
AND STATUS = 'IN'; 




EMPLOYEE_ID,PUNCH_TIME,STATUS
103,2025-06-26T08:31:00.000Z,IN



#####Task 6:

Write an SQL query to calculate the total number of hours each employee spends inside the office

In [0]:
%sql

WITH TIME AS (
SELECT EMPLOYEE_ID,
      STATUS,
      LEAD(STATUS) OVER (PARTITION BY EMPLOYEE_ID ORDER BY PUNCH_TIME) AS NEXT_STATUS,
      PUNCH_TIME,
      LEAD(PUNCH_TIME) OVER (PARTITION BY EMPLOYEE_ID ORDER BY PUNCH_TIME) AS NEXT_PUNCH_TIME
FROM PUNCH_LOG
ORDER BY EMPLOYEE_ID, PUNCH_TIME)

SELECT EMPLOYEE_ID,
       DATEDIFF(HOUR, PUNCH_TIME, NEXT_PUNCH_TIME) AS TIME_SPEND_IN_OFFICE
FROM TIME
WHERE STATUS = 'IN'
AND NEXT_STATUS = 'OUT';

EMPLOYEE_ID,TIME_SPEND_IN_OFFICE
101,9
102,3
103,9
104,8
105,6
